In the following set of experiments, we aim to generate meaningful benchmarks for the dose escalation methods we would like to study.

### Utilities

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
from doseescalation.dose_escalator import (
    CRMDoseEscalator, 
    DoseEscalatorBase,
    ThreePlusThreeDoseEscalator, 
    UCBDoseEscalator,
    SEEDADoseEscalator,
    SEEDAPlateauDoseEscalator,
    SEEDAPlateauFixedDoseEscalator,
    SEEDAPlateauOriginalDoseEscalator
)
from doseescalation.estimator import (
    AveragingEstimator
)
from doseescalation.evaluate import (
    plot_dose_proposals, 
    plot_acc_progression,
    plot_n_dles,
    simulate
)
from doseescalation.simulated_env import SimulatedEnv
from typing import Callable, Sequence

/opt/anaconda3/envs/ucl/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
# Get the current timestamp for saving results:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [3]:
def a_key(a):
    return f"a = {a:.1f}"

def cohort_key(cohort):
    return f"Cohort {cohort + 1}"

In [4]:
def dose_toxic_curve(dose_levels, a_hat):
    return np.power((np.tanh(dose_levels) + 1) / 2, a_hat)

def inv_dose_toxic(p_dle, a):
    return np.arctanh(2 * np.power(p_dle, 1 / a) - 1)

In [5]:
N_LEVELS = 6
P_DLE_LEVELS = {
    0.4: [0.3, 0.4, 0.53, 0.62, 0.76, 0.87],
    1.0: [0.05, 0.1, 0.2, 0.3, 0.5, 0.7],
    1.3: [0.02, 0.05, 0.12, 0.3, 0.41, 0.63],
    3.4: [0.01, 0.02, 0.04, 0.08, 0.16, 0.3]
}
DOSE_LEVELS = {
    a: [inv_dose_toxic(v, a) for v in vs] 
    for a, vs in P_DLE_LEVELS.items()
}
N_A = len(DOSE_LEVELS.keys())
TTL = 0.32
N_TRIALS = 300

# To add more algorithms, extend the following list to include them.
ALGOS = ["3 + 3", "CRM", "UCB", "SEEDA", "SEEDA Plateau", "SEEDA Plateau Fixed", "SEEDA Plateau Original"]

COHORT_SIZE = 3
CORRECT_MTDS = {
    a_key(0.4): 0, 
    a_key(1.0): 3, 
    a_key(1.3): 3, 
    a_key(3.4): 5
}

# UCB parameter:
UCB_COEFF = 0.1

# SEEDA and SEEDA Plateau parameters:
P_HAT = (0.05, 0.15, 0.3, 0.35, 0.4, 0.45)
Q_HAT = np.array([0.33] * 6)
ETA = 2
# Number of patients who showed positive response must be <= COHORT_SIZE (≈ 33% efficacy rate):
N_EFFICATE = 1

# Dose-efficacy curve is the per-patient response ~ Bernoulli(q_dose),
# rising then plateauing (shared across all toxicity scenarios):
EFFICACY_PROBS = [0.1, 0.35, 0.6, 0.6, 0.6, 0.6]

# Efficacy env: reuse SimulatedEnv with "dose levels" = indices so the curve maps
# index -> probability, giving n_efficate ~ Binomial(cohort, EFFICACY_PROBS[idx]).
EFFICACY_ENV = SimulatedEnv(list(range(N_LEVELS)), lambda i: EFFICACY_PROBS[int(i)])

# Algorithms that consume efficacy: score these against the efficacy-optimal
# dose (plateau onset), not the toxicity MTD.
EFFICACY_ALGOS = ("SEEDA", "SEEDA Plateau", "SEEDA Plateau Fixed", "SEEDA Plateau Original")

def _optimal_dose(tox_probs, eff_probs, ttl):
    # k* = lowest safe dose achieving the max efficacy among the safe doses:
    safe = [k for k in range(len(tox_probs)) if tox_probs[k] <= ttl]
    if not safe:
        return 0
    best = max(eff_probs[k] for k in safe)
    return min(k for k in safe if eff_probs[k] == best)

OPTIMAL_DOSES = {
    a_key(a): _optimal_dose(P_DLE_LEVELS[a], EFFICACY_PROBS, TTL)
    for a in P_DLE_LEVELS
}

# Per-(scenario, algorithm) correct dose, for the plots: efficacy-aware methods
# are judged against the efficacy-optimal dose, the rest against the toxicity MTD.
CORRECT_DOSES = {
    a_key(a): {
        algo: (OPTIMAL_DOSES[a_key(a)] if algo in EFFICACY_ALGOS
               else CORRECT_MTDS[a_key(a)])
        for algo in ALGOS
    } for a in DOSE_LEVELS
}

In [6]:
def run_simulations(
    dose_escalator: DoseEscalatorBase,
    dose_levels: Sequence[float],
    dose_toxic_curve: Callable,
    cohort_size: int, 
    n_cohorts: int,
    n_efficate: int = 0,
    efficacy_env=None,
):
    env = SimulatedEnv(dose_levels, dose_toxic_curve)
    return simulate(
        cohort_sizes=[cohort_size] * n_cohorts, 
        dose_escalator=dose_escalator, 
        env=env,
        n_efficate=n_efficate,
        efficacy_env=efficacy_env
    )

In [7]:
def build_results_table(rec_map, alloc_map, correct_mtds, n_levels, algos,
                        opt_doses=None, efficacy_algos=()):
    records = []
    for scenario in rec_map:
        for algo in algos:
            # Efficacy-aware methods are scored against the efficacy-optimal dose:
            if opt_doses is not None and algo in efficacy_algos:
                correct = opt_doses[scenario]
            # Toxicity-only methods are scored against the toxicity MTD:
            else:
                correct = correct_mtds[scenario]
            recs = np.asarray(rec_map[scenario][algo])
            allocs = np.asarray(alloc_map[scenario][algo])
            for dose in range(n_levels):
                records.append({
                    "Scenario": scenario,
                    "Algorithm": algo,
                    "Dose": dose,
                    # Whether this dose is the algorithm's correct (target) dose:
                    "Is correct": dose == correct,
                    "Rec (in %)": round(100 * np.mean(recs == dose), 2) if recs.size else np.nan,
                    "Alloc (in %)": round(100 * np.mean(allocs == dose), 2) if allocs.size else np.nan
                })
    return pd.DataFrame.from_records(records)

In [8]:
def highlight_correct_dose(pivot_df, correct_doses_dict):
    """
    Returns a Styler for a pivot table with index (Scenario, Algorithm) and
    columns = dose levels. Highlights the correct-dose cell per row in green.
    """
    styles = pd.DataFrame('', index=pivot_df.index, columns=pivot_df.columns)
    for (scenario, algo) in pivot_df.index:
        correct = correct_doses_dict[scenario][algo]
        if correct in pivot_df.columns:
            styles.loc[(scenario, algo), correct] = 'background-color: #2ca02c82; font-weight: bold'  # cooked asparagus green
    return pivot_df.style.apply(lambda _: styles, axis=None)

### Simulations

We run 2 main sets of simulations, differing in the number of cohorts used in their experiments. 

The more realistic setting uses around 10^1 cohorts while the setting for asymptotic behaviours uses more than 10^2.

#### Realistic

In [9]:
N_REAL_COHORTS = 10

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [10]:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_REAL_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

def add_simulations(dose_escalator, a, algo, n_efficate=0, n_cohorts=N_REAL_COHORTS):
    allocations, recommendations, n_dles = run_simulations(
        dose_escalator,
        dose_levels,
        lambda dose: dose_toxic_curve(dose, a),
        COHORT_SIZE,
        n_cohorts=n_cohorts,
        n_efficate=n_efficate,
        efficacy_env=EFFICACY_ENV
    )
    # Determine the final declared MTD for this trial:
    a_algo_rec_map[a_key(a)][algo].append(recommendations[-1])

    # Determine every cohort's allocated dose pooled across trials:
    a_algo_alloc_map[a_key(a)][algo].extend(allocations)

    # Determine the total toxicities during this trial:
    a_algo_n_dle_map[a_key(a)][algo].append(sum(n_dles))

    # Determine the recommendation at each cohort:
    for cohort, rec in enumerate(recommendations):
        cohort_algo_rec_map[a_key(a)][cohort_key(cohort)][algo].append(rec)

# Run the realistic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0])
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1])
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2])
        
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE)

        seedapl_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_dose_escalator, a, ALGOS[4], N_EFFICATE)

        seedapl_fixed_dose_escalator = SEEDAPlateauFixedDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_fixed_dose_escalator, a, ALGOS[5], N_EFFICATE)

        seedapl_original_dose_escalator = SEEDAPlateauOriginalDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_original_dose_escalator, a, ALGOS[6], N_EFFICATE)

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [11]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Realistic/Recommendations/real_algo_dose_rec_proposals.png"
)

Plot the proposals made for each cohort, so that we can see the time evolution of our `DoseEscalator`'s proposals. This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [12]:
for a in DOSE_LEVELS.keys():
    correct_mtds = {
        cohort_key(cohort): CORRECT_DOSES[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    mtds = {
        cohort_key(cohort): CORRECT_MTDS[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    plot_dose_proposals(
        N_LEVELS, 
        N_TRIALS, 
        cohort_algo_rec_map[a_key(a)], 
        correct_mtds, 
        mtds=mtds,
        unit_width=100,
        title_text="MTD recommendations at each cohort",
        show_fig=False,
        img_path=f"plots/{timestamp}/Realistic/Recommendations/real_{a_key(a)}_dose_rec_proposal_progression.png"
    )

Plot the distribution of dose limiting events across all the trial runs and cohorts.

In [13]:
plot_n_dles(
    a_algo_n_dle_map, 
    img_path=f"plots/{timestamp}/Realistic/real_algo_n_dles.png"
)

Plot the distribution of dose allocations (the dose each cohort actually received), pooled across all cohorts and trial runs.

In [14]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_REAL_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Realistic/Allocations/real_algo_dose_allocations.png"
)

Make a table with the metrics.

In [15]:
real_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, CORRECT_MTDS, N_LEVELS, ALGOS,
    opt_doses=OPTIMAL_DOSES, efficacy_algos=EFFICACY_ALGOS
)

In [16]:
# Paper-style wide layout (one column per dose level):
real_rec_pivot = real_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Rec (in %)", sort=False
)
real_alloc_pivot = real_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Alloc (in %)", sort=False
)

# Correct-dose summary: for each scenario/algorithm, the % recommending and
# allocating that algorithm's correct dose (efficacy-optimal for SEEDA/Plateau,
# toxicity MTD otherwise), plus whether that correct dose is the toxicity MTD.
correct_only = real_table[real_table["Is correct"]].copy()
correct_only["Is MTD"] = [
    dose == CORRECT_MTDS[scenario]
    for scenario, dose in zip(correct_only["Scenario"], correct_only["Dose"])
]
correct_summary = correct_only.set_index(["Scenario", "Algorithm"])[
    ["Dose", "Is MTD", "Rec (in %)", "Alloc (in %)"]
].rename(columns={
    "Dose": "Correct dose",
    "Rec (in %)": "Correct dose rec %",
    "Alloc (in %)": "Correct dose alloc %",
})

print("Recommendation % by dose level")
display(highlight_correct_dose(real_rec_pivot, CORRECT_DOSES))
print("\nAllocation % by dose level")
display(highlight_correct_dose(real_alloc_pivot, CORRECT_DOSES))
print("\nCorrect-dose summary")
display(correct_summary)

Recommendation % by dose level



Allocation % by dose level



Correct-dose summary


Correct dose  Is MTD  Correct dose rec %  \
Scenario Algorithm                                                          
a = 0.4  3 + 3                              0    True               83.00   
         CRM                                0    True               45.33   
         UCB                                0    True               37.00   
         SEEDA                              0    True               37.00   
         SEEDA Plateau                      0    True               96.00   
         SEEDA Plateau Fixed                0    True               93.33   
         SEEDA Plateau Original             0    True               56.00   
a = 1.0  3 + 3                              3    True               21.33   
         CRM                                3    True               25.33   
         UCB                                3    True               19.33   
         SEEDA                              2   False               53.00   
         SEEDA Plateau                      2   False                0.67   
         SEEDA Plateau Fixed                2   False               17.00   
         SEEDA Plateau Original             2   False               18.33   
a = 1.3  3 + 3                              3    True               30.67   
         CRM                                3    True               27.33   
         UCB                                3    True               20.67   
         SEEDA                              2   False               67.00   
         SEEDA Plateau                      2   False                0.00   
         SEEDA Plateau Fixed                2   False               14.33   
         SEEDA Plateau Original             2   False               36.00   
a = 3.4  3 + 3                              5    True                4.33   
         CRM                                5    True               33.33   
         UCB                                5    True               22.33   
         SEEDA                              2   False               25.67   
         SEEDA Plateau                      2   False                0.33   
         SEEDA Plateau Fixed                2   False               14.00   
         SEEDA Plateau Original             2   False                1.33   

                                 Correct dose alloc %  
Scenario Algorithm                                     
a = 0.4  3 + 3                                  80.70  
         CRM                                    47.63  
         UCB                                    23.37  
         SEEDA                                  33.90  
         SEEDA Plateau                          33.63  
         SEEDA Plateau Fixed                    34.03  
         SEEDA Plateau Original                 33.23  
a = 1.0  3 + 3                                  16.07  
         CRM                                    17.57  
         UCB                                    18.90  
         SEEDA                                  34.90  
         SEEDA Plateau                          38.40  
         SEEDA Plateau Fixed                    38.97  
         SEEDA Plateau Original                 36.10  
a = 1.3  3 + 3                                  23.80  
         CRM                                    21.07  
         UCB                                    19.20  
         SEEDA                                  35.87  
         SEEDA Plateau                          39.80  
         SEEDA Plateau Fixed                    41.13  
         SEEDA Plateau Original                 38.70  
a = 3.4  3 + 3                                  14.00  
         CRM                                    17.67  
         UCB                                    21.40  
         SEEDA                                  20.97  
         SEEDA Plateau                          32.70  
         SEEDA Plateau Fixed                    33.80  
         SEEDA Plateau Original                 26.57

Export the realistic tables as images.

In [17]:
"""import os
import nest_asyncio
import dataframe_image as dfi

nest_asyncio.apply()  # needed for dfi's Playwright backend inside Jupyter's asyncio loop

tables_path = f"plots/{timestamp}/Realistic/Tables"
os.makedirs(tables_path, exist_ok=True)

dfi.export(highlight_correct_dose(real_rec_pivot, CORRECT_DOSES).format("{:.2f}"),
           f"{tables_path}/real_rec_by_dose.png", dpi=150)
dfi.export(highlight_correct_dose(real_alloc_pivot, CORRECT_DOSES).format("{:.2f}"),
           f"{tables_path}/real_alloc_by_dose.png", dpi=150)
dfi.export(correct_summary.style.format({"Correct dose rec %": "{:.2f}", "Correct dose alloc %": "{:.2f}"}),
           f"{tables_path}/real_correct_dose_summary.png", dpi=150)
print(f"Saved realistic tables to {tables_path}/")"""

'import os\nimport nest_asyncio\nimport dataframe_image as dfi\n\nnest_asyncio.apply()  # needed for dfi\'s Playwright backend inside Jupyter\'s asyncio loop\n\ntables_path = f"plots/{timestamp}/Realistic/Tables"\nos.makedirs(tables_path, exist_ok=True)\n\ndfi.export(highlight_correct_dose(real_rec_pivot, CORRECT_DOSES).format("{:.2f}"),\n           f"{tables_path}/real_rec_by_dose.png", dpi=150)\ndfi.export(highlight_correct_dose(real_alloc_pivot, CORRECT_DOSES).format("{:.2f}"),\n           f"{tables_path}/real_alloc_by_dose.png", dpi=150)\ndfi.export(correct_summary.style.format({"Correct dose rec %": "{:.2f}", "Correct dose alloc %": "{:.2f}"}),\n           f"{tables_path}/real_correct_dose_summary.png", dpi=150)\nprint(f"Saved realistic tables to {tables_path}/")'

#### Asymptotic

In [18]:
N_ASYM_COHORTS = 300

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [19]:
# Reinitialise the maps for the asymptotic setting:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_ASYM_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

# Run the asymptotic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0], n_cohorts=N_ASYM_COHORTS)
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1], n_cohorts=N_ASYM_COHORTS)
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2], n_cohorts=N_ASYM_COHORTS)
        
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        seedapl_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_dose_escalator, a, ALGOS[4], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        seedapl_fixed_dose_escalator = SEEDAPlateauFixedDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_fixed_dose_escalator, a, ALGOS[5], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        seedapl_original_dose_escalator = SEEDAPlateauOriginalDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_original_dose_escalator, a, ALGOS[6], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [20]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_algo_dose_rec_proposals.png",
)

Plot the distribution of dose allocations, pooled across all cohorts and trial runs.

In [21]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_ASYM_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Asymptotic/Allocations/asym_algo_dose_allocations.png",
)

Plot the proposal accuracy (whether it matches the MTD) for each cohort so that we can see the time evolution of our `DoseEscalator`'s proposals. 

This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [22]:
plot_acc_progression(
    N_ASYM_COHORTS, 
    cohort_algo_rec_map,
    CORRECT_DOSES,
    show_fig=False, 
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_dose_rec_proposal_acc_progression.png",
)

Make a table with the metrics.

In [23]:
asym_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, CORRECT_MTDS, N_LEVELS, ALGOS,
    opt_doses=OPTIMAL_DOSES, efficacy_algos=EFFICACY_ALGOS,
)

In [24]:
# Paper-style wide layout (one column per dose level):
asym_rec_pivot = asym_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Rec (in %)", sort=False
)
asym_alloc_pivot = asym_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Alloc (in %)", sort=False
)

# Correct-dose summary: for each scenario/algorithm, the % recommending and
# allocating that algorithm's correct dose (efficacy-optimal for SEEDA/Plateau,
# toxicity MTD otherwise), plus whether that correct dose is the toxicity MTD.
correct_only = asym_table[asym_table["Is correct"]].copy()
correct_only["Is MTD"] = [
    dose == CORRECT_MTDS[scenario]
    for scenario, dose in zip(correct_only["Scenario"], correct_only["Dose"])
]
correct_summary = correct_only.set_index(["Scenario", "Algorithm"])[
    ["Dose", "Is MTD", "Rec (in %)", "Alloc (in %)"]
].rename(columns={
    "Dose": "Correct dose",
    "Rec (in %)": "Correct dose rec %",
    "Alloc (in %)": "Correct dose alloc %",
})

print("Recommendation % by dose level")
display(highlight_correct_dose(asym_rec_pivot, CORRECT_DOSES))
print("\nAllocation % by dose level")
display(highlight_correct_dose(asym_alloc_pivot, CORRECT_DOSES))
print("\nCorrect-dose summary")
display(correct_summary)

Recommendation % by dose level



Allocation % by dose level



Correct-dose summary


Correct dose  Is MTD  Correct dose rec %  \
Scenario Algorithm                                                          
a = 0.4  3 + 3                              0    True               87.00   
         CRM                                0    True               50.33   
         UCB                                0    True               68.67   
         SEEDA                              0    True               70.00   
         SEEDA Plateau                      0    True              100.00   
         SEEDA Plateau Fixed                0    True              100.00   
         SEEDA Plateau Original             0    True              100.00   
a = 1.0  3 + 3                              3    True               20.33   
         CRM                                3    True               28.00   
         UCB                                3    True                6.00   
         SEEDA                              2   False               55.67   
         SEEDA Plateau                      2   False                0.00   
         SEEDA Plateau Fixed                2   False               98.67   
         SEEDA Plateau Original             2   False                5.67   
a = 1.3  3 + 3                              3    True               26.00   
         CRM                                3    True               35.00   
         UCB                                3    True               10.00   
         SEEDA                              2   False               59.33   
         SEEDA Plateau                      2   False                0.00   
         SEEDA Plateau Fixed                2   False               97.33   
         SEEDA Plateau Original             2   False                8.33   
a = 3.4  3 + 3                              5    True                0.00   
         CRM                                5    True               31.00   
         UCB                                5    True                7.67   
         SEEDA                              2   False               25.00   
         SEEDA Plateau                      2   False                0.00   
         SEEDA Plateau Fixed                2   False               86.00   
         SEEDA Plateau Original             2   False                0.00   

                                 Correct dose alloc %  
Scenario Algorithm                                     
a = 0.4  3 + 3                                  86.90  
         CRM                                    50.29  
         UCB                                    62.32  
         SEEDA                                  93.27  
         SEEDA Plateau                          94.09  
         SEEDA Plateau Fixed                    94.10  
         SEEDA Plateau Original                 93.32  
a = 1.0  3 + 3                                  20.18  
         CRM                                    27.69  
         UCB                                     7.97  
         SEEDA                                  47.30  
         SEEDA Plateau                          49.68  
         SEEDA Plateau Fixed                    50.31  
         SEEDA Plateau Original                 48.90  
a = 1.3  3 + 3                                  25.82  
         CRM                                    34.68  
         UCB                                    11.64  
         SEEDA                                  47.84  
         SEEDA Plateau                          49.42  
         SEEDA Plateau Fixed                    49.53  
         SEEDA Plateau Original                 49.12  
a = 3.4  3 + 3                                   0.47  
         CRM                                    30.53  
         UCB                                     9.39  
         SEEDA                                  23.67  
         SEEDA Plateau                          26.70  
         SEEDA Plateau Fixed                    27.94  
         SEEDA Plateau Original                 24.04

Export the asymptotic tables as images.

In [25]:
"""tables_path = f"plots/{timestamp}/Asymptotic/Tables"
os.makedirs(tables_path, exist_ok=True)

dfi.export(highlight_correct_dose(asym_rec_pivot, CORRECT_DOSES).format("{:.2f}"),
           f"{tables_path}/asym_rec_by_dose.png", dpi=150)
dfi.export(highlight_correct_dose(asym_alloc_pivot, CORRECT_DOSES).format("{:.2f}"),
           f"{tables_path}/asym_alloc_by_dose.png", dpi=150)
dfi.export(correct_summary.style.format({"Correct dose rec %": "{:.2f}", "Correct dose alloc %": "{:.2f}"}),
           f"{tables_path}/asym_correct_dose_summary.png", dpi=150)
print(f"Saved asymptotic tables to {tables_path}/")"""

'tables_path = f"plots/{timestamp}/Asymptotic/Tables"\nos.makedirs(tables_path, exist_ok=True)\n\ndfi.export(highlight_correct_dose(asym_rec_pivot, CORRECT_DOSES).format("{:.2f}"),\n           f"{tables_path}/asym_rec_by_dose.png", dpi=150)\ndfi.export(highlight_correct_dose(asym_alloc_pivot, CORRECT_DOSES).format("{:.2f}"),\n           f"{tables_path}/asym_alloc_by_dose.png", dpi=150)\ndfi.export(correct_summary.style.format({"Correct dose rec %": "{:.2f}", "Correct dose alloc %": "{:.2f}"}),\n           f"{tables_path}/asym_correct_dose_summary.png", dpi=150)\nprint(f"Saved asymptotic tables to {tables_path}/")'